# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR² Kilifi Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Identifier: {metadata.identifier}\nVersion: {metadata.version}\nLicense: {metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The Croissant dataset schema organizes data using `RecordSet` entities. Each `RecordSet` contains `Field` objects, each with a unique `@id`. We'll inspect the record sets, fields, and columns and reference them by their `@id`s.

In [ ]:
# List available record sets by `@id`
record_sets = dataset.record_sets
print("Available record sets:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# Show fields and columns in the first record set
if record_sets:
    first_rs_id = record_sets[0]['@id']
    fields = record_sets[0].get('field', [])
    print("\nFields in the first record set:")
    for field in fields:
        print(f"- Field @id: {field['@id']}, name: {field.get('name', 'N/A')}, dataType: {field.get('dataType', 'N/A')}")

    columns = record_sets[0].get('column', [])
    print("\nColumns in the first record set:")
    for column in columns:
        print(f"- Column @id: {column['@id']}, name: {column.get('name', 'N/A')}, dataType: {column.get('dataType', 'N/A')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. We use the record set and field `@id`s from the above overview.

We'll load all available record sets. If multiple, each will be shown. Fields and columns always referenced by `@id`.

In [ ]:
# Extract data from each record set using @id
dataframes = {}

for rs in record_sets:
    rs_id = rs['@id']
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records for RecordSet @id: {rs_id}")
        print(f"Columns (@id): {df.columns.tolist()}\n")
    except Exception as e:
        print(f"Failed to load records for {rs_id}: {e}")

# Display the first few records from the primary record set
if record_sets:
    main_rs_id = record_sets[0]['@id']
    print(f"Sample rows from RecordSet (ID: {main_rs_id}):")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Typical EDA operations:
- Remove outliers
- Transform data distributions
- Group data by a key attribute

Here, we select a numeric field (such as GAD-7 score, PHQ-9 score, or age), filter records, normalize values, and group by another field (e.g. village or gender). All fields referenced by their `@id`.

In [ ]:
# Example: Filter and normalize GAD-7 score, group by 'village' or demographic field
# Replace these with actual @id values identified in the data overview above.

main_rs_id = record_sets[0]['@id']
df = dataframes[main_rs_id]

# Identify field @ids, example: 'gad7_score', 'age', 'village'
numeric_field_id = None
group_field_id = None
for col in df.columns:
    if 'gad7' in col.lower():
        numeric_field_id = col
    if 'village' in col.lower():
        group_field_id = col

if numeric_field_id is None:
    # fallback if GAD-7 not present
    for col in df.columns:
        if 'phq9' in col.lower():
            numeric_field_id = col

if numeric_field_id is None:
    for col in df.columns:
        if 'age' in col.lower():
            numeric_field_id = col

if group_field_id is None:
    for col in df.columns:
        if 'gender' in col.lower():
            group_field_id = col

print(f"Selected numeric field @id: {numeric_field_id}")
print(f"Selected group field @id: {group_field_id}")

# Convert numeric field to numeric dtype
df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

# Filter records where score > threshold (for demonstration, threshold = 10)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
display(filtered_df.head())

# Normalize the score in this subset
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group field and show mean
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot histograms of the selected numeric field and a boxplot grouped by the group field using their `@id`s.

In [ ]:
# Simple histogram of the numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field_id].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Boxplot grouped by group field (if present)
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We successfully loaded and explored the FAIR² Kilifi Mental Health Survey dataset using `mlcroissant`.
- Record sets and fields are referenced via their `@id`, supporting reproducible and schema-driven analyses.
- Numeric fields such as GAD-7, PHQ-9, and demographic attributes can be analyzed for distribution, normalization, and grouped statistics.
- Visual EDA revealed sample trends in the survey responses; further insights could be derived for public health and research use cases.